In [ ]:
# @title 🚀 शिव एआई - ड्राइव और गिटहब मास्टर कंट्रोल
from google.colab import drive, output
from transformers import AutoModelForCausalLM, AutoTokenizer
from IPython.display import display, HTML, Audio
from gtts import gTTS
import torch, os, ipywidgets as widgets

# १. ड्राइव और मालिक की पहचान
OWNER = "Shri Ram nag"
DRIVE_PATH = "/content/drive/MyDrive/ShivAI_Backup"

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

print("⌛ शिव एआई का दिमाग ड्राइव से लोड हो रहा है...")
tokenizer = AutoTokenizer.from_pretrained(DRIVE_PATH)
model = AutoModelForCausalLM.from_pretrained(DRIVE_PATH, torch_dtype=torch.bfloat16, device_map="auto")
print("✅ गिटहब और ड्राइव सिंक सफल!")

In [ ]:
# @title 🧠 वॉइस और चैट इंटरफेस (माइक फिक्स)
def speak_fixed(text):
    # हकलाहट फिक्स: नंबर को शब्दों में बदलें
    n_map = {"1":"एक", "2":"दो", "3":"तीन", "4":"चार", "5":"पाँच", "0":"शून्य", ".":"डॉट"}
    for n, w in n_map.items():
        text = text.replace(n, w)
    tts = gTTS(text, lang='hi')
    filename = "shiv_voice.wav"
    tts.save(filename)
    return filename

def ai_engine(user_in):
    sys_msg = f"You are Shiv AI by {OWNER}. Answer shortly in Hindi."
    prompt = f"<|system|>\n{sys_msg}<|user|>\n{user_in}<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    # टर्बो जनरेशन और रिपीटेशन फिक्स
    out_ids = model.generate(**inputs, max_new_tokens=80, repetition_penalty=1.2, temperature=0.4, do_sample=True)
    return tokenizer.decode(out_ids[0], skip_special_tokens=True).split("<|assistant|>\n")[-1].strip()

ui_code = HTML(f'''
<style>
    .card {{ background: #1a1a1a; color: #FF4500; padding: 25px; border-radius: 20px; text-align: center; border: 3px solid #FF4500; box-shadow: 0 0 15px #FF4500; font-family: sans-serif; }}
    .v-btn {{ width: 60px; height: 60px; background: #007bff; border: none; border-radius: 50%; color: white; font-size: 24px; cursor: pointer; transition: 0.3s; }}
    .v-btn.active {{ background: #f44336; transform: scale(1.1); box-shadow: 0 0 10px #f44336; }}
</style>
<div class="card">
    <h2>🚩 शिव एआई (Shiv AI)</h2>
    <p><b>मालिक:</b> {OWNER}</p>
    <button id="mic" class="v-btn" onclick="startListen()">🎤</button>
    <p id="msg">सुनने के लिए तैयार हूँ...</p>
</div>
<script>
    function startListen() {{
        var btn = document.getElementById('mic');
        var r = new (window.SpeechRecognition || window.webkitSpeechRecognition)();
        r.lang = 'hi-IN';
        r.onstart = () => {{ btn.classList.add('active'); document.getElementById('msg').innerText='सुन रहा हूँ...'; }};
        r.onresult = (e) => {{ google.colab.kernel.invokeFunction('notebook.process', [e.results[0][0].transcript], {{}}); }};
        r.onend = () => {{ btn.classList.remove('active'); document.getElementById('msg').innerText='तैयार!'; }};
        r.start();
    }}
</script>\n''')

chat_out = widgets.Output()
def process(txt):
    with chat_out:
        print(f"👤 आप: {txt}")
        ans = ai_engine(txt)
        print(f"🚩 शिव एआई: {ans}")
        display(Audio(speak_fixed(ans), autoplay=True))

output.register_callback('notebook.process', process)
display(ui_code, chat_out)